# Mini-MIAS: Normal vs Abnormal Classification

**Pipeline:** Whole-mammogram -> CLAHE -> MobileNetV2 (ImageNet, frozen) features -> StandardScaler -> RBF-SVM

**Task (Option 2):** binary classification of the **full 322-image** Mini-MIAS database into:
- `0 = NORM` (normal, 207 images)
- `1 = ABNORMAL` (benign or malignant lesion, 115 images)

**Key fixes vs the old 111-image notebook:**
1. Uses **all 322 images** (normal images have no ROI, so we classify the *whole* mammogram for both classes -> consistent preprocessing, no train/test mismatch).
2. **Split is done by image BEFORE augmentation** -> no data leakage (rotations of one image never span train & test).
3. Reports **5-fold cross-validation** + a held-out test set, with sensitivity/specificity, ROC-AUC and confusion matrix.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# ---------------- Config ----------------
IMAGE_DIR = '/content/drive/MyDrive/Colab/all-mias/'
INFO_PATH = '/content/drive/MyDrive/Colab/all-mias/Info1.txt'
SAVE_DIR  = '/content/drive/MyDrive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE   = 224
ROT_ANGLES = [0, 45, 90, 135, 180, 225, 270, 315]  # train-only augmentation
BATCH_SIZE = 16
RANDOM_STATE = 2021

## 1. Parse Info1.txt -> one label per image (322 images)

Each image is `NORM` (normal) or has an abnormality with severity `B`/`M`.
We collapse to **Normal (0)** vs **Abnormal (1)**. Multi-lesion images (e.g. mdb005) are counted once.

In [ ]:
def build_label_table(info_path):
    rows = {}
    with open(info_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        img_id = parts[0]
        if not img_id.startswith('mdb'):
            continue  # skip header / stray lines
        cls = parts[2]

        if cls == 'NORM':
            label, sev = 0, 'N'
        else:
            sev = parts[3] if len(parts) > 3 else ''
            if sev not in ('B', 'M'):
                continue
            label = 1  # abnormal (benign or malignant)

        # one row per image; abnormal overrides if an image somehow appears as both
        if img_id not in rows or label == 1:
            rows[img_id] = {'image_id': img_id, 'label': label, 'severity': sev}

    df = pd.DataFrame(rows.values()).sort_values('image_id').reset_index(drop=True)
    return df

df = build_label_table(INFO_PATH)
print('Total unique images:', len(df))
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Abnormal'}))
df.head()

## 2. Preprocessing (whole image, consistent for both classes)

Grayscale PGM -> resize 224 -> CLAHE -> normalize -> stack to 3 channels (for ImageNet backbone).

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_and_preprocess(img_id):
    path = os.path.join(IMAGE_DIR, img_id + '.pgm')
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = _clahe.apply(img)
    img = img / 255.0
    return np.stack([img] * 3, axis=-1).astype(np.float32)

def rotate(img, angle):
    if angle == 0:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT_101)

## 3. Image-wise split BEFORE augmentation (prevents leakage)

In [ ]:
# keep only images whose .pgm actually exists
df = df[df['image_id'].apply(lambda i: os.path.exists(os.path.join(IMAGE_DIR, i + '.pgm')))].reset_index(drop=True)
print('Images with files present:', len(df))

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df['label'],
    random_state=RANDOM_STATE,
)
print('Train images:', len(train_df), '| Test images:', len(test_df))
print('Train dist:', train_df['label'].value_counts().to_dict())
print('Test  dist:', test_df['label'].value_counts().to_dict())

In [ ]:
def build_set(frame, augment):
    X, Y = [], []
    angles = ROT_ANGLES if augment else [0]
    for _, row in frame.iterrows():
        base = load_and_preprocess(row['image_id'])
        if base is None:
            continue
        for a in angles:
            X.append(rotate(base, a))
            Y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(Y)

x_train, y_train = build_set(train_df, augment=True)   # augmented
x_test,  y_test  = build_set(test_df,  augment=False)  # original only

print('x_train:', x_train.shape, '| class counts:', np.bincount(y_train))
print('x_test :', x_test.shape,  '| class counts:', np.bincount(y_test))

## 4. MobileNetV2 feature extractor (frozen)

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

feature_extractor = Sequential([base_model, GlobalAveragePooling2D()])

def extract_features(model, data):
    feats = []
    for i in range(0, len(data), BATCH_SIZE):
        feats.append(model.predict(data[i:i + BATCH_SIZE], verbose=0))
    return np.vstack(feats)

train_feat = extract_features(feature_extractor, x_train)
test_feat  = extract_features(feature_extractor, x_test)
print('train_feat:', train_feat.shape, '| test_feat:', test_feat.shape)

## 5. Scale + train SVM

In [ ]:
scaler = StandardScaler()
train_feat_s = scaler.fit_transform(train_feat)
test_feat_s  = scaler.transform(test_feat)

svm = SVC(C=10, gamma='scale', kernel='rbf',
          probability=True, class_weight='balanced',
          random_state=42)
svm.fit(train_feat_s, y_train)
print('SVM trained.')

## 6. Evaluate on held-out test set

In [ ]:
y_pred = svm.predict(test_feat_s)
y_prob = svm.predict_proba(test_feat_s)[:, 1]

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('ROC-AUC :', round(roc_auc_score(y_test, y_prob), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Abnormal']))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) else 0
specificity = tn / (tn + fp) if (tn + fp) else 0
print('Confusion matrix [ [TN FP] [FN TP] ]:\n', cm)
print('Sensitivity (recall abnormal):', round(sensitivity, 4))
print('Specificity (recall normal)  :', round(specificity, 4))

## 7. 5-fold cross-validation (image-level, augment train fold only)

This is the number to report in a paper: mean +/- std over folds, with **no leakage** because folds are split on images before augmentation.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ids = df['image_id'].values
labels = df['label'].values

acc_scores, auc_scores, sens_scores, spec_scores = [], [], [], []

for fold, (tr_idx, te_idx) in enumerate(skf.split(ids, labels), 1):
    tr_frame = df.iloc[tr_idx]
    te_frame = df.iloc[te_idx]

    Xtr, ytr = build_set(tr_frame, augment=True)
    Xte, yte = build_set(te_frame, augment=False)

    ftr = extract_features(feature_extractor, Xtr)
    fte = extract_features(feature_extractor, Xte)

    sc = StandardScaler().fit(ftr)
    ftr, fte = sc.transform(ftr), sc.transform(fte)

    clf = SVC(C=10, gamma='scale', kernel='rbf',
              probability=True, class_weight='balanced', random_state=42)
    clf.fit(ftr, ytr)

    p  = clf.predict(fte)
    pr = clf.predict_proba(fte)[:, 1]
    cm = confusion_matrix(yte, p)
    tn, fp, fn, tp = cm.ravel()

    acc_scores.append(accuracy_score(yte, p))
    auc_scores.append(roc_auc_score(yte, pr))
    sens_scores.append(tp / (tp + fn) if (tp + fn) else 0)
    spec_scores.append(tn / (tn + fp) if (tn + fp) else 0)
    print(f'Fold {fold}: acc={acc_scores[-1]:.3f} auc={auc_scores[-1]:.3f} '
          f'sens={sens_scores[-1]:.3f} spec={spec_scores[-1]:.3f}')

def ms(v):
    return f'{np.mean(v):.3f} +/- {np.std(v):.3f}'

print('\n===== 5-fold CV (mean +/- std) =====')
print('Accuracy   :', ms(acc_scores))
print('ROC-AUC    :', ms(auc_scores))
print('Sensitivity:', ms(sens_scores))
print('Specificity:', ms(spec_scores))

## 8. ROC curve (held-out test set)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as auc_fn

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc_fn(fpr, tpr)

plt.figure(figsize=(6, 6), dpi=150)
plt.plot(fpr, tpr, color='#c0392b', lw=2,
         label=f'MobileNetV2 + SVM (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('1 - Specificity (False Positive Rate)')
plt.ylabel('Sensitivity (True Positive Rate)')
plt.title('ROC - Mini-MIAS Normal vs Abnormal')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'roc_normal_vs_abnormal.png'), dpi=300, bbox_inches='tight')
plt.show()

## 9. Ablation study (CLAHE on/off, augmentation on/off)

Re-runs the full 5-fold CV pipeline under four settings so you can quantify each component's contribution. Report this as an ablation table in the paper.

In [ ]:
def load_opt(img_id, use_clahe):
    path = os.path.join(IMAGE_DIR, img_id + '.pgm')
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    if use_clahe:
        img = _clahe.apply(img)
    img = img / 255.0
    return np.stack([img] * 3, axis=-1).astype(np.float32)

def build_set_opt(frame, use_clahe, use_augment):
    X, Y = [], []
    angles = ROT_ANGLES if use_augment else [0]
    for _, row in frame.iterrows():
        base = load_opt(row['image_id'], use_clahe)
        if base is None:
            continue
        for a in angles:
            X.append(rotate(base, a))
            Y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(Y)

def run_cv(use_clahe, use_augment):
    accs, aucs = [], []
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for tr_idx, te_idx in skf2.split(ids, labels):
        Xtr, ytr = build_set_opt(df.iloc[tr_idx], use_clahe, use_augment)
        Xte, yte = build_set_opt(df.iloc[te_idx], use_clahe, False)  # never augment test
        ftr = extract_features(feature_extractor, Xtr)
        fte = extract_features(feature_extractor, Xte)
        sc = StandardScaler().fit(ftr)
        ftr, fte = sc.transform(ftr), sc.transform(fte)
        clf = SVC(C=10, gamma='scale', kernel='rbf', probability=True,
                  class_weight='balanced', random_state=42).fit(ftr, ytr)
        p  = clf.predict(fte)
        pr = clf.predict_proba(fte)[:, 1]
        accs.append(accuracy_score(yte, p))
        aucs.append(roc_auc_score(yte, pr))
    return np.mean(accs), np.std(accs), np.mean(aucs), np.std(aucs)

settings = [
    ('No CLAHE, No Aug',   False, False),
    ('CLAHE, No Aug',      True,  False),
    ('No CLAHE, Aug',      False, True),
    ('CLAHE + Aug (full)', True,  True),
]

results = []
for name, clh, aug in settings:
    am, asd, um, usd = run_cv(clh, aug)
    results.append({'Setting': name,
                    'Accuracy': f'{am:.3f} +/- {asd:.3f}',
                    'ROC-AUC':  f'{um:.3f} +/- {usd:.3f}'})
    print(f'{name:22s} | acc {am:.3f}+/-{asd:.3f} | auc {um:.3f}+/-{usd:.3f}')

ablation_df = pd.DataFrame(results)
ablation_df.to_csv(os.path.join(SAVE_DIR, 'ablation_results.csv'), index=False)
ablation_df

## 10. Confusion matrix & auto-generated results summary

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix - Held-out Test Set')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_normal_vs_abnormal.png'),
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Auto-generate a paper-ready results paragraph from the computed metrics.
# Run AFTER sections 6 (hold-out), 7 (CV) and 9 (ablation).
n_total = len(df)
n_norm  = int((df['label'] == 0).sum())
n_abn   = int((df['label'] == 1).sum())

summary = (
    f"On the full Mini-MIAS database (N = {n_total} images; "
    f"{n_norm} normal, {n_abn} abnormal), a MobileNetV2 feature extractor "
    f"(ImageNet-pretrained, frozen) combined with an RBF-SVM classified mammograms "
    f"as normal vs. abnormal. Using image-wise stratified 5-fold cross-validation "
    f"(augmentation applied to the training folds only, to avoid data leakage), the "
    f"model achieved an accuracy of {np.mean(acc_scores)*100:.1f}% "
    f"(+/- {np.std(acc_scores)*100:.1f}), ROC-AUC of {np.mean(auc_scores):.3f} "
    f"(+/- {np.std(auc_scores):.3f}), sensitivity of {np.mean(sens_scores)*100:.1f}% "
    f"and specificity of {np.mean(spec_scores)*100:.1f}%. On an independent 20% "
    f"hold-out test set, the model reached {accuracy_score(y_test, y_pred)*100:.1f}% "
    f"accuracy and an ROC-AUC of {roc_auc_score(y_test, y_prob):.3f} "
    f"(sensitivity {sensitivity*100:.1f}%, specificity {specificity*100:.1f}%)."
)

print(summary)

with open(os.path.join(SAVE_DIR, 'results_summary.txt'), 'w') as f:
    f.write(summary)
print('\nSaved -> results_summary.txt')

## 11. Save model + scaler

In [ ]:
joblib.dump(svm,    os.path.join(SAVE_DIR, 'svm_mias_normal_vs_abnormal.pkl'))
joblib.dump(scaler, os.path.join(SAVE_DIR, 'scaler_mias_normal_vs_abnormal.pkl'))
print('Saved to', SAVE_DIR)